# 🎓 Educational Deep Dive: Slowly Changing Dimensions (SCD)

Welcome to the SCD Framework bootcamp! 
When data changes in the source system (e.g., a customer moves to a new city), how do we reflect that in our Data Warehouse? Do we overwrite it? Do we keep a history? 
This is what **SCD (Slowly Changing Dimensions)** solves. We will explore 4 strategies using Delta Lake's powerful `MERGE` command.

Let's master the art of time-travel in data!

# Databricks Data Engineering Bootcamp
## Peripheral Module A: Mastering Slowly Changing Dimensions (SCD Type 0, 1, 2, 3)

**Context:** Managing historical mutations in master registries is a core responsibility of a Data Engineer. Depending on downstream analytical requirements, you must implement the correct SCD pattern:
1. **SCD Type 0 (Retain Original):** Attributes are immutable. Once written, incoming changes are completely ignored (e.g., Enrollment Dates).
2. **SCD Type 1 (Overwrite):** Updates values in place. No history is kept.
3. **SCD Type 3 (Add New Column):** Tracks limited history by storing the "Previous" and "Current" states side-by-side in the same row.
4. **SCD Type 2 (Add New Row):** Full history tracking via active status flags and validity date windows.

### 🔑 Architectural Alignment (Silver Layer Grain Constraint)
In compliance with enterprise data platform standards, all Slowly Changing Dimension transformations are executed strictly inside the Silver Layer. The Fact table grain remains completely protected because transactional tables will join against these standardized dimensional states.

**Task:** You will deploy all four SCD engines using PySpark and Delta Lake to analyze how an incoming change affects your physical tables.

### Chapter 1: Pipeline Baseline Setup
We simulate our base target state (Day 1) and the incoming daily delta updates (Day 2).

### HASH DIFF EXPLANATION
For every row, we generate two distinct hash keys using a hashing function (like SHA-256):
1. The Business Hash Key (based strictly on the Customer ID).
2. The Full Row Hash Key (or Hash Diff), which is based on all the columns combined.

How it works in practice:
The next day, if a record with the same Customer ID arrives, the Business Hash Key will remain exactly the same.
However, if the same ID arrives but the customer's city has changed, the Business Hash Key stays identical, 
but the Full Row Hash Key (Hash Diff) will be completely different. 
We simply compare the Full Row Hash Keys to instantly detect if a record was updated!

In [0]:
%sql
USE CATALOG ai_lab;
USE SCHEMA default;

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, LongType, StringType

# Read the REAL table we created in Notebook 1
target_df = spark.read.table("silver_dim_customers")

# Dynamically get the very first real customer ID and Name to ensure an update happens
real_cust_id = target_df.select("customer_id").first()[0]
real_cust_name = target_df.select("customer_name").first()[0]

# Create the new batch: The old customer changed city, and we added a brand new one
incoming_df = spark.createDataFrame([
    (real_cust_id, real_cust_name, "Patras (MOVED!)"), # This will trigger an UPDATE
    (999999, "Eleni", "Thessaloniki")                  # This will trigger an INSERT
], ["customer_id", "customer_name", "city"])

print("Incoming Delta Staging Batch (Day 2):")
display(incoming_df)

## Chapter 2: Strategy A — SCD Type 0: Retain Original (Static Master Data)

### SCD Type 0 Mechanics: Immutable Records
In SCD Type 0, attributes are considered completely static. Once a record is inserted for a specific customer_id, any downstream mutations or updates in the source staging system are explicitly ignored. 

![SCD Type 0](scd_type0.png)

### 👨‍🏫 Instructor Note: SCD Type 0 (Retain Original)
**Concept:** Once a record is written, it is *never* changed. Updates are completely ignored. 
**How we solved it:** We replaced the blank with `whenNotMatchedInsert`. This tells Delta Lake: *"If the ID doesn't exist, insert it. If it already exists, do absolutely nothing."*

In [0]:
# 1. Clone the real table to protect it
spark.sql("CREATE OR REPLACE TABLE silver_dim_customers_type0 CLONE silver_dim_customers")

In [0]:
from delta.tables import DeltaTable

# --- SCD TYPE 0: INSERT-ONLY MERGE ---
print("SCD Type 0: The following data WOULD be merged (Insert-Only logic):")
display(incoming_df)

(
    # Connect to the cloned table
    DeltaTable.forName(spark, "silver_dim_customers_type0").alias("target")
    # Compare incoming data with existing table based on ID
    .merge(incoming_df.alias("source"), "target.customer_id = source.customer_id")
    # If customer is completely new, insert it. Ignore any updates.
    .whenNotMatchedInsert(values = {
        "customer_id": "source.customer_id", 
        "customer_name": "source.customer_name", 
        "city": "source.city"
    })
    .execute()
)

## Chapter 3: Strategy B — SCD Type 1: Overwrite (No History Tracking)

### SCD Type 1 Mechanics: Inline Overwrite
SCD Type 1 updates historical attribute states in place. When a change arrives, the old value is permanently deleted and overwritten with the active state. No history is retained.

![SCD Type 1](scd_type1.png)

### 👨‍🏫 Instructor Note: SCD Type 1 (Overwrite)
**Concept:** We don't care about history. We always want the most up-to-date value.
**How we solved it:** Inside the `whenMatchedUpdate` clause, we simply map the target's `"city"` to the incoming `"source.city"`. The old city is lost forever.

In [0]:
# 1. Clone the real table to protect it
spark.sql("CREATE OR REPLACE TABLE silver_dim_customers_type1 CLONE silver_dim_customers")

In [0]:
# --- SCD TYPE 1: OVERWRITE MERGE ---
print("SCD Type 1: The following data WOULD be merged (Overwrite logic):")
display(incoming_df)

(
    # Connect to the cloned table
    DeltaTable.forName(spark, "silver_dim_customers_type1").alias("target")
    .merge(incoming_df.alias("source"), "target.customer_id = source.customer_id")
    # If customer exists, overwrite their existing data with the new city
    .whenMatchedUpdate(set = {
        "customer_name": "source.customer_name",
        "city": "source.city" 
    })
    # If customer is new, insert them
    .whenNotMatchedInsert(values = {
        "customer_id": "source.customer_id", 
        "customer_name": "source.customer_name", 
        "city": "source.city"
    })
    .execute()
)

## Chapter 4: Strategy C — SCD Type 3 (Current vs Previous Column Tracking)

### SCD Type 3 Mechanics: Column-Level Dual Horizon Tracking
SCD Type 3 tracks a limited historical horizon by shifting values into dedicated parallel columns inside the same row. This keeps the schema light while allowing analysts to compare the "Current" vs "Previous" state directly.

![SCD Type 3](scd_type3.png)

### 👨‍🏫 Instructor Note: SCD Type 3 (Current & Previous)
**Concept:** We want *some* history, but we don't want multiple rows per customer. We add a dedicated column for the old value.
**How we solved it:** When an update arrives, we first take the `target.current_city` and save it into the `previous_city` field. Then we update the `current_city` with the new incoming value.

In [0]:
# 1. Clone the real table and add the 'previous_city' column for tracking
spark.sql("CREATE OR REPLACE TABLE silver_dim_customers_type3 CLONE silver_dim_customers")
spark.sql("ALTER TABLE silver_dim_customers_type3 ADD COLUMNS (previous_city STRING)")

In [0]:
# --- SCD TYPE 3: PREVIOUS & CURRENT STATE MERGE ---
print("SCD Type 3: The following data WOULD be merged (Keep Previous State logic):")
display(incoming_df)

(
    # Connect to the cloned table
    DeltaTable.forName(spark, "silver_dim_customers_type3").alias("target")
    .merge(incoming_df.alias("source"), "target.customer_id = source.customer_id")
    # If the customer exists and the city changed, shift the old city to previous_city
    .whenMatchedUpdate(
        condition = "target.city <> source.city",
        set = {
            "previous_city": "target.city", 
            "city": "source.city"            
        }
    )
    # If new customer, insert them and set previous_city to null
    .whenNotMatchedInsert(values = {
        "customer_id": "source.customer_id",
        "customer_name": "source.customer_name",
        "city": "source.city",
        "previous_city": "null" 
    })
    .execute()
)

## Chapter 5: Strategy D — SCD Type 2 (Full Row Historization)

### SCD Type 2 Mechanics: Full Multi-Row Auditing & Versioning
SCD Type 2 is the gold standard for enterprise dimensional historization. Every single change prompts a split: the active record is closed out (is_current = False, end_date = current_timestamp), and a brand new record row is appended to house the active state.

![SCD Type 2](scd_type2.png)

### 👨‍🏫 Instructor Note: SCD Type 2 (Full Row Historization)
**Concept:** The Enterprise Standard. We keep *every* change by creating a new row for the new state, and "closing" the old row using `is_current` flags and `start/end` dates.
**How we solved it:** 
1. First, we identify that an update is needed.
2. We use `whenMatchedUpdate` to set `"is_current": "false"` and timestamp the `end_date` on the old row.
3. The `whenNotMatchedInsert` automatically inserts the new row with `is_current = true`.

In [0]:
# 1. Clone the real table and add Type 2 tracking columns
spark.sql("CREATE OR REPLACE TABLE silver_dim_customers_type2 CLONE silver_dim_customers")
spark.sql("ALTER TABLE silver_dim_customers_type2 ADD COLUMNS (is_current BOOLEAN, start_date TIMESTAMP, end_date TIMESTAMP)")

# Initialize the old records so they appear as "Active" currently
spark.sql("UPDATE silver_dim_customers_type2 SET is_current = true, start_date = current_timestamp()")

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

# 2. Add tracking metadata to the incoming data
updates_with_meta = (
    incoming_df.withColumn("is_current", F.lit(True))
    .withColumn("start_date", F.current_timestamp())
    .withColumn("end_date", F.lit(None).cast("timestamp"))
)

# 3. Create a 'mergeKey' to trick the Delta engine into performing both UPDATE and INSERT
staged_updates = (
    # First part: Find existing customers and create a row with a NULL mergeKey
    # This NULL forces the MERGE statement below to treat this row as an "Insert"!
    updates_with_meta.join(spark.read.table("silver_dim_customers_type2").filter("is_current = true"), "customer_id", "inner")
    .select(
        updates_with_meta["*"],
        F.lit(None).cast("long").alias("mergeKey") # The "Fake" ID to force an Insert
    )
    # Second part: Combine with the original incoming data where mergeKey is the real ID
    # This REAL ID forces the MERGE statement below to treat this row as an "Update"
    .unionByName(
        updates_with_meta.withColumn("mergeKey", F.col("customer_id")) 
    )
)

print("SCD Type 2: The following formatted data WOULD be merged (Full History Tracking):")
display(staged_updates)

# 4. Execute Type 2 Merge Transaction
(
    DeltaTable.forName(spark, "silver_dim_customers_type2").alias("target")
    # IMPORTANT: We compare the target ID with our custom mergeKey (NOT the source customer_id)
    .merge(staged_updates.alias("source"), "target.customer_id = source.mergeKey")
    
    # If the real ID matched, it means we need to close the old historical record
    .whenMatchedUpdate(
        condition = "target.is_current = true AND target.city <> source.city",
        set = {
            "is_current": "false",
            "end_date": "source.start_date" 
        }
    )
    # If the mergeKey was NULL (or a brand new customer), we insert the new record
    .whenNotMatchedInsert(values = {
        "customer_id": "source.customer_id", # <-- Here we insert the REAL ID, not the mergeKey!
        "customer_name": "source.customer_name",
        "city": "source.city",
        "is_current": "source.is_current",
        "start_date": "source.start_date",
        "end_date": "source.end_date"
    })
    .execute()
)